In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from jaxtyping import Array, Scalar
from tqdm import tqdm
from utils import (
    init_linear_weight,
    ProgressPlotter,
)

This notebook is adapted from the original notebook by Ben Moseley https://github.com/benmoseley/harmonic-oscillator-pinn-workshop

# PINN for the harmonic oscillator
We have the harmonic oscillator:

$$
\ddot{u}_{\theta} + 2 \sigma \dot{u}_{\theta} + \omega^2 u_{\theta} = 0
$$

where $\sigma$ is the damping coefficient and $\omega$ is the natural frequency

In [ ]:
sigma2 = 3.5
omega = 30
sample_rate = 1000
dt = 1.0 / sample_rate
n_steps = 1000
u0 = 0.75
v0 = 0.0


def complex_oscillator(
    sigma2: float,
    omega: float,
    dt: float,
    n_steps: int,
    u0: float = 1.0,
    v0: float = 0.0,
):
    """
    Complex oscillator with damping.
    This function simulates a damped harmonic oscillator using complex exponentials.
    """

    sigma = sigma2 / 2
    omega_damped = np.sqrt(omega**2 - sigma**2)
    s_mu = -sigma + 1j * omega_damped
    z_mu = np.exp(s_mu * dt)
    C = u0 - ((v0 + sigma * u0) / omega_damped) * 1j
    return C * z_mu ** np.arange(n_steps)


complex_sol = complex_oscillator(
    sigma2=sigma2,
    omega=omega,
    dt=dt,
    n_steps=n_steps,
    u0=u0,
    v0=v0,
)

Generate some sparse training data

In [ ]:
t = jnp.arange(n_steps) * dt
t_data = t[0 : n_steps // 3 : 10]
w_data = complex_sol.real[0 : n_steps // 3 : 10]

Define the PINN model

In [ ]:
class PINN(eqx.Module):
    sigma2: Array
    omega: Array
    B: Array
    mlp: eqx.nn.MLP

    def fourier_features(self, x: Array) -> Array:
        """Apply Fourier feature encoding to inputs."""
        return jnp.concatenate(
            [jnp.cos(2 * jnp.pi * self.B @ x), jnp.sin(2 * jnp.pi * self.B @ x)]
        )

    def __init__(
        self,
        sigma2: Array,
        omega: Array,
        n_fourier: int,
        fourier_scale_t: float,
        key: Scalar,
    ):
        self.B = jax.random.normal(key, (n_fourier, 1)) * fourier_scale_t

        mlp = eqx.nn.MLP(
            in_size=2 * n_fourier,
            out_size="scalar",
            depth=3,
            width_size=128,
            activation=jax.nn.tanh,
            key=jax.random.split(loss_key, 1)[0],
        )

        self.mlp = init_linear_weight(
            mlp,
            jax.nn.initializers.glorot_uniform(),
            jax.random.split(loss_key, 1)[0],
        )

        self.sigma2 = sigma2
        self.omega = jnp.exp(omega)

    def __call__(self, x: Array) -> Array:
        """Forward pass with Fourier feature encoding."""
        # Ensure x has at least 1 dimension for matrix multiplication
        x = jnp.atleast_1d(x)
        fourier_x = self.fourier_features(x)
        return self.mlp(fourier_x)


loss_key = jax.random.PRNGKey(3407)

## PINN Loss Function

In [ ]:
def loss_fn(
    model: eqx.Module,
    x: Array,  # input time (N,)
    y: Array,  # target displacement (N,)
    key: Scalar,
):
    # Forward pass through the model
    y_pred = jax.vmap(model)(x)  # shape (N,)

    # data loss
    data_loss = jnp.mean((y_pred - y) ** 2)

    # physical loss
    x_physical = jax.random.uniform(key, (256,), minval=0, maxval=t[-1])
    y_physical = jax.vmap(model)(x_physical)
    dt_ = jax.vmap(jax.grad(model))(x_physical)
    dtt = jax.vmap(jax.grad(jax.grad(model)))(x_physical)
    residual = dtt + model.sigma2 * dt_ + model.omega**2 * y_physical
    physical_loss = jnp.mean(residual**2)

    # initial condition losses
    y_ic_pred = model(0.0)
    y_ic_loss = (y_ic_pred - u0) ** 2

    dt_ic = jax.grad(model)(0.0)
    dt_ic_loss = (dt_ic - v0) ** 2

    loss = (
        1e-4 * physical_loss + 1e-0 * y_ic_loss + 1e-0 * dt_ic_loss + 1e-1 * data_loss
    )

    return loss

In [ ]:

model = PINN(
    sigma2=jnp.array([5.0]),  # Initial guess for sigma
    omega=jnp.log(jnp.array([20.0])),  # Initial guess for omega_mu_squared
    n_fourier=64,
    fourier_scale_t=5,
    key=loss_key,
)


@eqx.filter_jit
def training_step(
    model: eqx.Module,
    x: Array,
    y: Array,
    optimizer: optax.GradientTransformation,
    opt_state: optax.OptState,
    key: Scalar,
):
    loss_value, grads = eqx.filter_value_and_grad(loss_fn)(
        model,
        x,
        y,
        key,
    )
    updates, opt_state = optimizer.update(
        grads,
        opt_state,
        eqx.filter(model, eqx.is_array),
    )
    model = eqx.apply_updates(model, updates)
    key = jax.random.split(key, 1)[0]

    return model, opt_state, loss_value, key


epochs = 50_000
schedule = optax.cosine_onecycle_schedule(
    transition_steps=epochs,
    peak_value=2e-3,
)
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(schedule),
)
opt_state = optimizer.init(
    eqx.filter(model, eqx.is_array),
)

# Set up progress plotting
progress_plotter = ProgressPlotter(
    output_dir="tmp_pinn", model_name="PINN_Oscillator", framerate=15
)

with tqdm(range(epochs), desc="Training PINN") as pbar:
    for epoch in pbar:
        # Perform a training step
        model, opt_state, loss_value, loss_key = training_step(
            model,
            t_data,
            w_data,
            optimizer,
            opt_state,
            loss_key,
        )

        # Update progress bar with loss and parameters
        gmu = model.sigma2.item()
        omu = model.omega.item()
        pbar.set_postfix(
            {
                "Loss": f"{loss_value:.4e}",
                "σ": f"{gmu/2:.2f}/{sigma2/2:.2f}",
                "ω": f"{omu:.2f}/{omega:.2f}",
            }
        )

        # Save training frame using ProgressPlotter
        if epoch % 1000 == 0:
            progress_plotter.save_pinn_frame(
                model=model,
                epoch=epoch,
                time_array=t,
                true_solution=complex_sol.real,
                training_data_time=t_data,
                training_data_values=w_data,
                true_params={"sigma2": sigma2, "omega": omega},
                title="PINN Harmonic Oscillator",
                epoch_interval=1000
            )

In [ ]:
# Final results visualization and saving
print("Generating final visualization...")

# Generate predictions for the full time range
y_pred = jax.vmap(model)(t)

# Create final comparison plot
plt.figure(figsize=(10, 5))

# Plot 1: Time series comparison
plt.subplot(2, 1, 1)
plt.plot(t, complex_sol.real, label="True Solution", linewidth=2)
plt.plot(t_data, w_data, "o", label="Training Data", markersize=4, alpha=0.7)
plt.plot(t, y_pred, "--", label="PINN Prediction", linewidth=2, alpha=0.8)
plt.xlabel("Time (s)")
plt.ylabel("Displacement")
plt.title("PINN vs Analytical Solution")
plt.legend()
plt.grid(True, alpha=0.3)


# Plot 2: Parameter results as text
plt.subplot(2, 1, 2)
plt.axis("off")

# Create text summary of parameters
sigma_error = abs(model.sigma2.item() / 2 - sigma2 / 2) / (sigma2 / 2) * 100
omega_error = abs(model.omega.item() - omega) / omega * 100

param_text = f"""Final Parameter Results:

True σ = {sigma2 / 2:.4f},  PINN σ = {model.sigma2.item() / 2:.4f}  (Error: {sigma_error:.1f}%)
True ω = {omega:.4f}, PINN ω = {model.omega.item():.4f} (Error: {omega_error:.1f}%)

Mean relative error: {jnp.mean(jnp.abs(y_pred - complex_sol.real) / jnp.abs(complex_sol.real)) * 100:.2f}%
"""

plt.text(
    0,
    0,
    param_text,
    transform=plt.gca().transAxes,
    fontsize=15,
    fontfamily="monospace",
)

plt.tight_layout()

# Save the plot
plt.savefig("pinn_results.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Generate training animation using ProgressPlotter
print("Generating training animation...")
progress_plotter.render_animation("pinn_training.webm")